# 07 - 升级分析结构和行号引用

这一节升级第 06 步，但仍然只分析 `AAPL_business_001`。主要变化：

- Python 先给原文行编号，模型只返回引用行号。
- 将原文事实 `facts` 和分析推断 `inferences` 分开。
- 将当前块缺少的信息命名为 `chunk_gaps_cn`，避免误认为整份财报没有披露。
- 保存 Prompt 版本、调用时间、Token 用量和精确引用位置。

> 模型调用单元格只发送一个 chunk，但执行一次仍会产生一次 DeepSeek API 请求。

## 1. 读取一个文本块

In [ ]:
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_deepseek import ChatDeepSeek
from pydantic import BaseModel, Field, field_validator, model_validator

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data/sec/AAPL"
CHUNK_ID = "AAPL_business_001"
PROMPT_VERSION = "filing-chunk-v2"
TEMPERATURE = 0

chunk_files = sorted(DATA_DIR.glob("*_chunks.json"), reverse=True)
if not chunk_files:
    raise FileNotFoundError("没有找到 chunks JSON，请先运行 05_chunk_sections.ipynb")

chunks_path = chunk_files[0]
chunks_data = json.loads(chunks_path.read_text(encoding="utf-8"))
selected_chunk = next(
    (chunk for chunk in chunks_data["chunks"] if chunk["chunk_id"] == CHUNK_ID),
    None,
)
if selected_chunk is None:
    raise ValueError(f"没有找到文本块：{CHUNK_ID}")

print(f"已选择：{selected_chunk['chunk_id']}")
print(f"章节：{selected_chunk['section_title']}")
print(f"字符数：{selected_chunk['char_count']:,}")

## 2. 由 Python 生成引用行号

Python 按自然句子边界把碎片化 HTML 文本合并成证据单元，每个单元获得稳定编号和精确位置。模型只能选择这些编号，不能自己编写引用文字。

In [ ]:
def build_evidence_lines(chunk: dict) -> list[dict]:
    text = chunk["text"]
    boundaries = list(
        re.finditer(r"(?<=[.!?])\s+(?=[A-Z0-9\"“])", text)
    )
    starts = [0, *[match.end() for match in boundaries]]
    ends = [*[match.start() for match in boundaries], len(text)]
    raw_ranges = [
        [start, end]
        for start, end in zip(starts, ends)
        if text[start:end].strip()
    ]

    # 将 Item 标题等很短的碎片并入下一个完整句子。
    merged_ranges = []
    index = 0
    while index < len(raw_ranges):
        start, end = raw_ranges[index]
        display_text = re.sub(r"\s+", " ", text[start:end]).strip()
        if len(display_text) < 40 and index + 1 < len(raw_ranges):
            end = raw_ranges[index + 1][1]
            index += 1
        merged_ranges.append((start, end))
        index += 1

    evidence_lines = []
    for start, end in merged_ranges:
        raw_text = text[start:end]
        leading_spaces = len(raw_text) - len(raw_text.lstrip())
        content = raw_text.strip()
        local_start = start + leading_spaces
        local_end = local_start + len(content)
        line_id = f"L{len(evidence_lines) + 1:03d}"

        evidence_lines.append({
            "line_id": line_id,
            "text": content,
            "display_text": re.sub(r"\s+", " ", content),
            "local_start": local_start,
            "local_end": local_end,
            "source_start": chunk["source_start"] + local_start,
            "source_end": chunk["source_start"] + local_end,
        })

    return evidence_lines


evidence_lines = build_evidence_lines(selected_chunk)
line_map = {line["line_id"]: line for line in evidence_lines}
numbered_text = "\n".join(
    f"[{line['line_id']}] {line['display_text']}" for line in evidence_lines
)

print(f"共生成 {len(evidence_lines)} 个句子级引用单元\n")
print("\n".join(numbered_text.splitlines()[:15]))

## 3. 定义升级后的数据结构

事实只能描述原文明确披露的内容；推断必须引用已生成的 `fact_id`，不能伪装成公司披露。

In [ ]:
FactCategory = Literal[
    "business_model",
    "product",
    "service",
    "geography",
    "distribution",
    "customer",
    "competition",
    "financial",
    "risk",
    "other",
]
Materiality = Literal["high", "medium", "low"]
Confidence = Literal["high", "medium", "low"]


def normalize_id(value: str | int, prefix: str) -> str:
    text = str(value).strip().upper()
    if text.startswith(prefix):
        text = text[len(prefix):]
    if text.isdigit():
        return f"{prefix}{int(text):03d}"
    return f"{prefix}{text}"


class FilingFact(BaseModel):
    fact_id: str = Field(description="当前块内唯一编号，例如 F001")
    category: FactCategory
    fact_cn: str = Field(description="原文明确支持的中文事实，不加入推断")
    evidence_line_ids: list[str] = Field(
        description="支持事实的一个或多个有效行号，例如 L003"
    )
    materiality: Materiality = Field(description="对公司研究的重要程度")

    @field_validator("fact_id", mode="before")
    @classmethod
    def normalize_fact_id(cls, value: str | int) -> str:
        return normalize_id(value, "F")

    @field_validator("category", mode="before")
    @classmethod
    def normalize_category(cls, value: str) -> str:
        aliases = {
            "业务模式": "business_model",
            "产品": "product",
            "服务": "service",
            "核心产品与服务": "other",
            "地区结构": "geography",
            "地域": "geography",
            "分销": "distribution",
            "客户": "customer",
            "竞争": "competition",
            "财务": "financial",
            "风险": "risk",
            "其他": "other",
        }
        text = str(value).strip()
        return aliases.get(text, text)

    @field_validator("evidence_line_ids", mode="before")
    @classmethod
    def normalize_line_ids(cls, value: list[str | int] | str | int) -> list[str]:
        values = value if isinstance(value, list) else [value]
        return [normalize_id(item, "L") for item in values]

    @field_validator("materiality", mode="before")
    @classmethod
    def normalize_materiality(cls, value: str) -> str:
        return {"高": "high", "中": "medium", "低": "low"}.get(
            str(value).strip(), str(value).strip()
        )

    @model_validator(mode="after")
    def refine_combined_category(self):
        if self.category == "other":
            if "服务包括" in self.fact_cn:
                self.category = "service"
            elif "产品" in self.fact_cn:
                self.category = "product"
        return self


class AnalystInference(BaseModel):
    inference_id: str = Field(description="当前块内唯一编号，例如 I001")
    inference_cn: str = Field(description="基于事实得出的分析判断，不是公司直接披露")
    based_on_fact_ids: list[str] = Field(description="支持该判断的 fact_id")
    confidence: Confidence

    @field_validator("inference_id", mode="before")
    @classmethod
    def normalize_inference_id(cls, value: str | int) -> str:
        return normalize_id(value, "I")

    @field_validator("based_on_fact_ids", mode="before")
    @classmethod
    def normalize_fact_ids(cls, value: list[str | int] | str | int) -> list[str]:
        values = value if isinstance(value, list) else [value]
        return [normalize_id(item, "F") for item in values]

    @field_validator("confidence", mode="before")
    @classmethod
    def normalize_confidence(cls, value: str) -> str:
        return {"高": "high", "中": "medium", "低": "low"}.get(
            str(value).strip(), str(value).strip()
        )


class ChunkAnalysisV2(BaseModel):
    chunk_id: str
    summary_cn: str
    facts: list[FilingFact] = Field(description="4 到 8 条重要事实")
    inferences: list[AnalystInference]
    chunk_gaps_cn: list[str] = Field(
        description="仅指当前文本块无法回答、可能在其他块披露的信息"
    )

    @field_validator("chunk_gaps_cn", mode="before")
    @classmethod
    def normalize_chunk_gaps(cls, value: list[str] | str) -> list[str]:
        return value if isinstance(value, list) else [value]

## 4. 根据章节设置分析重点

不同章节不能使用完全相同的关注点。Business 优先提取业务模式、产品、服务、地区、客户、分销和竞争信息。

In [ ]:
SECTION_PRIORITIES = {
    "business": (
        "优先提取业务模式、核心产品与服务、地区结构、客户、分销、竞争和关键依赖。"
        "除非对业务分析有实质影响，否则不要把财年日期等申报常规信息列为重要事实。"
    ),
    "risk_factors": (
        "优先提取风险事件、风险原因、潜在影响、风险集中度和公司披露的缓释措施。"
    ),
    "mda": (
        "优先提取期间变化、管理层解释的驱动因素、利润率、流动性、资本支出和关键估计。"
    ),
}
section_priority = SECTION_PRIORITIES[selected_chunk["section"]]

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """你是一名审慎的公司财报研究助手。
只能依据用户提供的 SEC 财报原文分析，不使用外部知识，不提供投资建议。
事实和分析推断必须分开：facts 只能写原文明确支持的事实，inferences 才能写分析判断。
inferences 也不得使用外部市场知识；原文没有价格、市场层级或竞争定位时，不得推断高端、中端、领先等结论。
每条事实只能引用输入中真实存在的行号，不得自行创建行号。
fact_id 必须使用 F001 格式，inference_id 必须使用 I001 格式，行号必须使用 L003 这样的完整字符串。
category 只能使用 business_model、product、service、geography、distribution、customer、competition、financial、risk、other。
materiality 和 confidence 只能使用 high、medium、low，不得翻译成中文。
evidence_line_ids、based_on_fact_ids、chunk_gaps_cn 即使只有一个值，也必须返回 JSON 数组。
chunk_gaps_cn 只描述当前文本块的缺口，不得声称整份财报没有披露。
必须只返回一个 JSON 对象，并包含 chunk_id、summary_cn、facts、inferences、chunk_gaps_cn。
每个 fact 必须包含 fact_id、category、fact_cn、evidence_line_ids、materiality。
每个 inference 必须包含 inference_id、inference_cn、based_on_fact_ids、confidence。""",
    ),
    (
        "human",
        """请分析下面的财报文本块。

chunk_id: {chunk_id}
ticker: {ticker}
form: {form}
section: {section_title}
分析重点: {section_priority}

<numbered_filing_text>
{numbered_text}
</numbered_filing_text>""",
    ),
])

## 5. 创建 JSON mode 分析链

使用 `include_raw=True` 同时保留解析后的 Pydantic 对象和模型原始响应元数据，以便记录 Token 用量。

In [ ]:
load_dotenv(PROJECT_ROOT / ".env", override=True)
api_key = os.getenv("DEEPSEEK_API_KEY", "").strip()
if not api_key:
    raise ValueError("请先在 .env 中配置 DEEPSEEK_API_KEY")

model_name = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
base_url = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
llm = ChatDeepSeek(
    model=model_name,
    api_key=api_key,
    base_url=base_url,
    temperature=TEMPERATURE,
    request_timeout=60,
    max_retries=2,
)
structured_llm = llm.with_structured_output(
    ChunkAnalysisV2,
    method="json_mode",
    include_raw=True,
)
analysis_chain = prompt | structured_llm

print(f"模型：{model_name}")
print(f"Prompt 版本：{PROMPT_VERSION}")
print("升级版分析链已创建")

## 6. 调用一次 DeepSeek

这个单元格会产生一次 API 请求。

In [ ]:
response = analysis_chain.invoke({
    "chunk_id": selected_chunk["chunk_id"],
    "ticker": selected_chunk["ticker"],
    "form": selected_chunk["form"],
    "section_title": selected_chunk["section_title"],
    "section_priority": section_priority,
    "numbered_text": numbered_text,
})

if response["parsing_error"] is not None:
    raise ValueError(f"结构化输出解析失败：{response['parsing_error']}")

analysis = response["parsed"]
raw_message = response["raw"]
analysis

## 7. 由 Python 解析和验证引用

Python 检查行号、事实编号和推断依赖，并从原文映射中生成真正的引文及精确位置。

In [ ]:
validation_errors = []
resolved_evidence = {}

if analysis.chunk_id != selected_chunk["chunk_id"]:
    validation_errors.append(f"chunk_id 不一致：{analysis.chunk_id}")

fact_ids = [fact.fact_id for fact in analysis.facts]
if len(fact_ids) != len(set(fact_ids)):
    validation_errors.append("fact_id 存在重复")

for fact in analysis.facts:
    evidence = []
    if not fact.evidence_line_ids:
        validation_errors.append(f"{fact.fact_id} 没有引用行号")

    for line_id in fact.evidence_line_ids:
        source_line = line_map.get(line_id)
        if source_line is None:
            validation_errors.append(f"{fact.fact_id} 使用了无效行号 {line_id}")
            continue
        evidence.append(source_line)

    resolved_evidence[fact.fact_id] = evidence

for inference in analysis.inferences:
    for fact_id in inference.based_on_fact_ids:
        if fact_id not in fact_ids:
            validation_errors.append(
                f"{inference.inference_id} 引用了不存在的事实 {fact_id}"
            )

if not 4 <= len(analysis.facts) <= 8:
    validation_errors.append("facts 数量不在 4 到 8 之间")

if validation_errors:
    print("校验未通过：")
    for error in validation_errors:
        print(f"- {error}")
else:
    print(f"校验通过：{len(analysis.facts)} 条事实的行号和依赖关系有效")

## 8. 显示事实、推断和块级缺口

In [ ]:
print(f"摘要：{analysis.summary_cn}\n")

print("事实：")
for fact in analysis.facts:
    print(f"[{fact.fact_id}] {fact.category} | {fact.materiality}")
    print(f"  {fact.fact_cn}")
    for line in resolved_evidence[fact.fact_id]:
        print(
            f"  [{line['line_id']}] {line['display_text']} "
            f"(source {line['source_start']}-{line['source_end']})"
        )

print("\n分析推断：")
for inference in analysis.inferences:
    print(
        f"[{inference.inference_id}] {inference.inference_cn} | "
        f"依据 {inference.based_on_fact_ids} | {inference.confidence}"
    )

print("\n当前块缺口：")
for gap in analysis.chunk_gaps_cn:
    print(f"- {gap}")

## 9. 保存升级版结果

旧版 `_analysis.json` 保留用于比较，新版保存为 `_analysis_v2.json`。

In [ ]:
usage_metadata = raw_message.usage_metadata or {}
response_metadata = raw_message.response_metadata or {}

output_data = {
    "run_metadata": {
        "prompt_version": PROMPT_VERSION,
        "analyzed_at_utc": datetime.now(timezone.utc).isoformat(),
        "requested_model": model_name,
        "response_model": response_metadata.get("model_name"),
        "temperature": TEMPERATURE,
        "structured_output_method": "json_mode",
        "token_usage": usage_metadata,
    },
    "source_chunks_file": str(chunks_path),
    "chunk_metadata": {
        key: selected_chunk[key]
        for key in (
            "chunk_id",
            "ticker",
            "form",
            "section",
            "source_start",
            "source_end",
        )
    },
    "model_output": analysis.model_dump(),
    "resolved_evidence": resolved_evidence,
    "validation": {
        "structure_passed": not validation_errors,
        "structure_errors": validation_errors,
        "inference_review_required": bool(analysis.inferences),
        "inference_review_status": "pending",
    },
}

output_path = DATA_DIR / f"{selected_chunk['chunk_id']}_analysis_v2.json"
output_path.write_text(
    json.dumps(output_data, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"升级版结果已保存：{output_path.resolve()}")

## 这一节完成了什么

升级版输出不再依赖模型复制英文引用。模型负责选择证据行和形成分析，Python 负责还原原文、位置和引用真实性。下一步需要用同一结构测试 Risk Factors 和 MD&A 各一个 chunk。